# Module 2 — Traffic Analyzer (Production)

This notebook is the **production pipeline** of VAAET. It combines real-time
perception (YOLO 11 + SORT tracking + speed estimation) with the trained
tabular classifier (TF/Keras MLP) to classify traffic states from video clips.

## Architecture

```mermaid
flowchart LR
    A[Video .mp4] --> B[YOLO 11\nDetection]
    B --> C[SORT\nTracking]
    C --> D[Speed\nEstimation]
    D --> E[Feature\nEngineering\n14 features]
    E --> F[MLP Classifier\ntraffic_classifier.keras]
    F --> G{Traffic State}
    G --> H[(telemetry_raw)]
    G --> I[(traffic_classifications)]
    H & I --> J[Feedback Loop\nRe-training]
    J -.-> F
```

## Traffic States

| State | Code | Criteria |
|---|---|---|
| **Normal** | 0 | Free flow (default) |
| **Reduced** | 1 | Degraded flow: 5–40 km/h, 15–25 veh/min |
| **Congested** | 2 | Congestion: <5 km/h, >25 veh/min, sustained ≥2 min |
| **Accident** | 3 | Disruptive event: ~0 km/h after sudden braking, sustained ≥3 min |

## Prerequisites

- **Trained model artifacts** — loaded automatically via 3-tier fallback:
  1. Local path `models/intelligence/` (same Colab session or local dev)
  2. Google Drive `MyDrive/vaaet/models/intelligence/` (persisted by M1)
  3. Manual upload widget (last resort on Colab)
- YOLO 11 weights (downloaded automatically at runtime)

In [ ]:
# Cell 0 — Environment Setup (Colab / Local)

import os
import sys

try:
    import google.colab  # type: ignore[import-untyped]
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = "https://github.com/titesen/vaaet.git"
    REPO_DIR = "/content/vaaet"
    NB_DIR = os.path.join(REPO_DIR, "notebooks", "02_production")

    if not os.path.isdir(REPO_DIR):
        print("📦 Cloning VAAET repository...")
        os.system(f"git clone --depth 1 {REPO_URL} {REPO_DIR}")
    else:
        print("📂 Repository found, pulling latest...")
        os.system(f"git -C {REPO_DIR} pull --ff-only")

    os.chdir(NB_DIR)
    sys.path.insert(0, REPO_DIR)
    print(f"✅ Colab CWD → {os.getcwd()}")
else:
    repo_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
    if repo_root not in sys.path:
        sys.path.insert(0, repo_root)
    print(f"✅ Local environment — CWD: {os.getcwd()}")

In [ ]:
# Cell 1 — Dependencies + Load Trained Model

import subprocess

def install_if_missing(package: str, import_name: str | None = None) -> None:
    name = import_name or package
    try:
        __import__(name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

install_if_missing("ultralytics")
install_if_missing("sqlalchemy")
install_if_missing("psycopg2-binary", "psycopg2")

import numpy as np
import pandas as pd
import cv2
import tensorflow as tf
import joblib
import shutil
from datetime import datetime

from src.config import (
    FEATURE_COLS, MODEL_DIR, MODEL_PATH, SCALER_PATH, LABEL_MAP_PATH,
    RANDOM_SEED, STATE_LABELS, MODEL_VERSION, VEHICLE_TYPES,
    DRIVE_ARTIFACT_DIR,
)
from src.features import engineer_features
from src.labeling import assign_traffic_state
from src.db import get_db_config, get_engine
from src.perception.detector import YOLODetector, select_model_variant
from src.perception.tracker import SORTTracker
from src.perception.speed import estimate_speed, is_stationary, SmoothedSpeedTracker
from src.perception.optical_flow import OpticalFlowEstimator
from src.video import validate_filename, extract_duration, open_video

np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

# Load trained artifacts — 3-tier fallback:
#   Tier 1: Local path (same session or local dev)
#   Tier 2: Google Drive (persisted by M1 across sessions)
#   Tier 3: Manual upload widget (last resort on Colab)

_root = os.path.join("..", "..")
_model_dir_abs = os.path.join(_root, MODEL_DIR)
os.makedirs(_model_dir_abs, exist_ok=True)

_ARTIFACT_NAMES = [
    os.path.basename(MODEL_PATH),      # traffic_classifier.keras
    os.path.basename(SCALER_PATH),     # feature_scaler.joblib
    os.path.basename(LABEL_MAP_PATH),  # label_mapping.joblib
]

def _local_paths() -> dict[str, str]:
    return {name: os.path.join(_model_dir_abs, name) for name in _ARTIFACT_NAMES}

def _all_present(paths: dict[str, str]) -> bool:
    return all(os.path.isfile(p) for p in paths.values())

_paths = _local_paths()
_tier = None

# ── Tier 1: Local ──
if _all_present(_paths):
    _tier = "local"

# ── Tier 2: Google Drive ──
if _tier is None and IN_COLAB:
    try:
        from google.colab import drive  # type: ignore[import-untyped]
        drive.mount("/content/drive", force_remount=False)
        _drive_dir = os.path.join("/content/drive", DRIVE_ARTIFACT_DIR)
        _drive_paths = {name: os.path.join(_drive_dir, name) for name in _ARTIFACT_NAMES}
        if _all_present(_drive_paths):
            for name in _ARTIFACT_NAMES:
                shutil.copy2(_drive_paths[name], os.path.join(_model_dir_abs, name))
            _paths = _local_paths()
            _tier = "Google Drive"
            print(f"📂 Artifacts copied from Drive → {_model_dir_abs}")
        else:
            _missing_drive = [n for n, p in _drive_paths.items() if not os.path.isfile(p)]
            print(f"⚠️  Drive checked but missing: {', '.join(_missing_drive)}")
    except Exception as e:
        print(f"⚠️  Drive mount skipped: {e}")

# ── Tier 3: Manual upload ──
if _tier is None and IN_COLAB:
    from google.colab import files as _colab_files  # type: ignore[import-untyped]
    print("📤 Upload trained artifacts (.keras + .joblib):")
    for name in _ARTIFACT_NAMES:
        if not os.path.isfile(os.path.join(_model_dir_abs, name)):
            print(f"   → {name}")
    _uploaded = _colab_files.upload()
    for fname, content in _uploaded.items():
        dest = os.path.join(_model_dir_abs, fname)
        with open(dest, "wb") as f:
            f.write(content)
    _paths = _local_paths()
    if _all_present(_paths):
        _tier = "upload"

# ── Load or fail ──
if _all_present(_paths):
    model = tf.keras.models.load_model(_paths[os.path.basename(MODEL_PATH)])
    scaler = joblib.load(_paths[os.path.basename(SCALER_PATH)])
    label_mapping = joblib.load(_paths[os.path.basename(LABEL_MAP_PATH)])
    print(f"✅ Dependencies loaded — TF {tf.__version__}")
    print(f"✅ Model loaded (via {_tier}): {_paths[os.path.basename(MODEL_PATH)]}")
    print(f"   Classes: {list(label_mapping.values())}")
else:
    print("🔴 Missing trained artifacts — run Module 1 (data_preparation) first:")
    for name, path in _paths.items():
        if not os.path.isfile(path):
            print(f"   ✗ {os.path.abspath(path)}")
    model = None
    scaler = None
    label_mapping = None

In [ ]:
# Cell 1b — Video Upload (Colab) or Path Selection (Local)

VIDEO_PATH: str | None = None

if IN_COLAB:
    from google.colab import files  # type: ignore[import-untyped]
    print("📤 Upload a video clip (.mp4):")
    uploaded = files.upload()
    if uploaded:
        VIDEO_PATH = list(uploaded.keys())[0]
        print(f"✅ Uploaded: {VIDEO_PATH}")
    else:
        print("⚠️ No file uploaded")
else:
    # Local: set the path manually or use a file dialog
    _default = os.path.join(_root, "data", "samples", "sample.mp4")
    VIDEO_PATH = _default
    print(f"📂 Local mode — set VIDEO_PATH manually or use default: {VIDEO_PATH}")
    print("   Example: VIDEO_PATH = r'C:\\path\\to\\bridge_2024-01-15_08-00-00_to_08-05-00.mp4'")

## Perception Pipeline

Two execution paths are available. **Use Cell 2b** for the full experience
(matches legacy notebook output):

| Path | Cell | Output | Download |
|---|---|---|---|
| **Full (recommended)** | **Cell 2b** | Annotated video + HUD + classification + `df_classified` | ✅ Auto-download popup on Colab |
| Telemetry only | Cell 2 → Cell 3 | `df_telemetry` → `df_classified` (no video) | — |

**Cell 2b** mirrors the legacy `process_bridge_video()`: it processes each
frame through YOLO 11 → SORT → speed estimation, renders the full HUD overlay,
classifies per minute, writes the annotated video, and triggers the browser
download dialog on Colab.

In [ ]:
# Cell 2 — Perception Pipeline (telemetry only, no video output)
#
# Use this cell INSTEAD of Cell 2b if you only need the per-minute
# telemetry DataFrame without generating an annotated video.
# If you want the full annotated video + download, SKIP this cell
# and run Cell 2b directly.
import time as _time

def _show_progress_bar(current: int, total: int, elapsed: float) -> None:
    """Legacy-style visual progress bar (█░) with ETA and fps."""
    pct = (current / total) * 100 if total else 0
    filled = int(40 * pct / 100)
    bar = "█" * filled + "░" * (40 - filled)
    if pct > 0 and elapsed > 0:
        eta = (elapsed / pct * 100) - elapsed
        eta_str = f"{int(eta // 60):02d}:{int(eta % 60):02d}"
    else:
        eta_str = "--:--"
    fps_proc = current / elapsed if elapsed > 0 else 0
    print(
        f"\r🎬 [{bar}] {pct:5.1f}% | Frame {current:,}/{total:,} | "
        f"⚡{fps_proc:.1f} fps | ETA: {eta_str}",
        end="", flush=True,
    )


def process_clip(video_path: str, model_variant: str | None = None) -> pd.DataFrame:
    """Process a video clip and extract per-minute telemetry.

    Pipeline per frame:
      1. YOLO 11 detects vehicles (filtered to 5 COCO classes).
      2. SORT tracker assigns persistent IDs via Euclidean centroid matching.
      3. Optical flow estimates global camera motion (pan/tilt compensation).
      4. Speed estimation (physics-based, perspective-corrected, camera-compensated).
      5. Stationary detection (AND-conjunction of 5 criteria).
      6. Count-once per unique track per minute (fixes legacy counting bug).

    Aggregation:
      Every ``frames_per_minute`` frames, a telemetry record is emitted with
      avg_speed, per-type vehicle counts, and total_vehicles.

    Args:
        video_path: Path to the .mp4 file.
        model_variant: YOLO model variant to use.  If ``None``, automatically
            selected based on clip duration via ``select_model_variant()``.

    Returns:
        DataFrame with one row per minute, matching ``traffic_data`` schema.
    """
    # Validate and open video
    if validate_filename(video_path):
        duration = extract_duration(video_path)
        print(f"📎 Valid bridge filename — duration: {duration:.0f}s")
    else:
        print("⚠️ Non-standard filename — extracting duration from metadata")
        try:
            duration = extract_duration(video_path)
        except ValueError:
            duration = 300.0  # fallback to 5 min
            print(f"⚠️ Could not determine duration, using {duration:.0f}s default")

    # Auto-select YOLO variant
    if model_variant is None:
        model_variant = select_model_variant(duration)
        print(f"🔍 Auto-selected model: {model_variant} (duration={duration:.0f}s)")

    # Initialize pipeline components
    detector = YOLODetector(model_variant=model_variant)
    detector.load()
    tracker = SORTTracker()
    flow_estimator = OpticalFlowEstimator()
    speed_tracker = SmoothedSpeedTracker(window_size=10)

    cap = open_video(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    frame_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frames_per_minute = int(fps * 60)

    records: list[dict] = []
    frame_idx = 0
    minute_counts: dict[str, int] = {vtype: 0 for vtype in VEHICLE_TYPES}
    minute_speeds: list[float] = []
    counted_tracks: set[int] = set()  # Track IDs already counted this minute
    cumulative_counts: dict[str, int] = {vtype: 0 for vtype in VEHICLE_TYPES}

    print(f"🎬 Processing: {os.path.basename(video_path)}")
    print(f"   {fps:.0f} FPS | {frame_h}p | ~{total_frames:,} frames | {model_variant}")
    print(f"📈 Progress:")

    wall_start = _time.time()
    last_progress_wall = wall_start

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # 1. Optical flow → global camera motion
        global_motion = flow_estimator.update(frame)

        # 2. Detect vehicles
        detections = detector.detect(frame)

        # 3. Track
        det_tuples = [(d.centroid, d.vehicle_type) for d in detections]
        active_tracks = tracker.update(det_tuples)

        # 4. Per-track: speed + stationary + count-once
        for track in active_tracks:
            # Speed estimation (with camera-motion compensation)
            speed = estimate_speed(
                track.history,
                fps=fps,
                frame_height=frame_h,
                global_motion=global_motion,
                vehicle_type=track.vehicle_type,
            )

            # Smoothed speed via SmoothedSpeedTracker
            smoothed = speed_tracker.update(track.track_id, speed)
            if smoothed is not None and not is_stationary(track.history):
                minute_speeds.append(smoothed)

            # Count-once: only tally a track the first time it's seen
            if track.track_id not in counted_tracks:
                if track.mark_counted():
                    minute_counts[track.vehicle_type] = (
                        minute_counts.get(track.vehicle_type, 0) + 1
                    )
                    counted_tracks.add(track.track_id)

        frame_idx += 1

        # ── Progress bar (every 2 s wall-clock) ──
        wall_now = _time.time()
        if wall_now - last_progress_wall >= 2.0:
            _show_progress_bar(frame_idx, total_frames, wall_now - wall_start)
            last_progress_wall = wall_now

        # ── Stats every 30 s of video time ──
        if frame_idx % int(fps * 30) == 0:
            cum_total = sum(cumulative_counts.values()) + sum(minute_counts.values())
            avg_spd = float(np.mean(minute_speeds)) if minute_speeds else 0.0
            active_n = len(active_tracks)
            print(f"\n📊 Stats @ {frame_idx / fps:.0f}s video time:")
            print(f"   📈 Total vehicles so far: {cum_total}")
            print(f"   ⚡ Current avg speed: {avg_spd:.1f} km/h")
            print(f"   🎯 Active tracks: {active_n}")
            for vt in VEHICLE_TYPES:
                c = cumulative_counts.get(vt, 0) + minute_counts.get(vt, 0)
                if c > 0:
                    print(f"   • {vt.upper()}: {c}")

        # Aggregate every minute
        if frame_idx % frames_per_minute == 0:
            avg_speed = float(np.mean(minute_speeds)) if minute_speeds else 0.0
            total = sum(minute_counts.values())
            records.append({
                "record_time": datetime.now(),
                "avg_speed": round(avg_speed, 2),
                "count_car": minute_counts.get("car", 0),
                "count_truck": minute_counts.get("truck", 0),
                "count_bus": minute_counts.get("bus", 0),
                "count_motorcycle": minute_counts.get("motorcycle", 0),
                "count_bicycle": minute_counts.get("bicycle", 0),
                "total_vehicles": total,
            })
            print(f"\n   📊 Minute {len(records)}: {avg_speed:.1f} km/h, {total} vehicles")

            # Accumulate into cumulative and reset per-minute
            for vt in VEHICLE_TYPES:
                cumulative_counts[vt] = cumulative_counts.get(vt, 0) + minute_counts.get(vt, 0)
            minute_counts = {vtype: 0 for vtype in VEHICLE_TYPES}
            minute_speeds.clear()
            counted_tracks.clear()

    # Flush remaining partial minute
    if frame_idx % frames_per_minute != 0 and (minute_speeds or any(minute_counts.values())):
        avg_speed = float(np.mean(minute_speeds)) if minute_speeds else 0.0
        total = sum(minute_counts.values())
        records.append({
            "record_time": datetime.now(),
            "avg_speed": round(avg_speed, 2),
            "count_car": minute_counts.get("car", 0),
            "count_truck": minute_counts.get("truck", 0),
            "count_bus": minute_counts.get("bus", 0),
            "count_motorcycle": minute_counts.get("motorcycle", 0),
            "count_bicycle": minute_counts.get("bicycle", 0),
            "total_vehicles": total,
        })
        for vt in VEHICLE_TYPES:
            cumulative_counts[vt] = cumulative_counts.get(vt, 0) + minute_counts.get(vt, 0)
        print(f"\n   📊 Partial minute {len(records)}: {avg_speed:.1f} km/h, {total} vehicles")

    cap.release()

    # ── Final summary (legacy style) ──
    _show_progress_bar(total_frames, total_frames, _time.time() - wall_start)
    wall_total = _time.time() - wall_start
    grand_total = sum(cumulative_counts.values())
    overall_fps = frame_idx / wall_total if wall_total > 0 else 0

    print(f"\n\n🎉 PROCESSING COMPLETE")
    print(f"⏱️  Wall time: {wall_total / 60:.1f} min")
    print(f"📊 Frames processed: {frame_idx:,}")
    print(f"🎯 Processing speed: {overall_fps:.1f} fps")
    print(f"📈 Detection summary:")
    for vt in VEHICLE_TYPES:
        c = cumulative_counts.get(vt, 0)
        if c > 0:
            print(f"   • {vt.upper()}: {c}")
    print(f"🚗 Total unique vehicles: {grand_total}")
    print(f"✅ Telemetry: {len(records)} minute(s)")

    return pd.DataFrame(records)


# Execution — uses VIDEO_PATH set by Cell 1b
try:
    if VIDEO_PATH and os.path.isfile(VIDEO_PATH):
        df_telemetry = process_clip(VIDEO_PATH)
    else:
        print("⚠️ No video file available. Set VIDEO_PATH in Cell 1b or upload a clip.")
        df_telemetry = None
except Exception as e:
    print(f"🔴 Error processing clip: {e}")
    df_telemetry = None

In [ ]:
# Cell 2b — Annotated Video Output with Full HUD + Classification
#
# This is the PRIMARY execution cell (mirrors legacy process_bridge_video).
# It processes the video, generates the annotated output with HUD,
# classifies each minute, triggers the Colab download popup, and sets
# df_telemetry / df_classified for downstream cells (4, 5, 6).

# ── Colour palette (matches legacy COLORS) ──
_COLORS: dict[str, tuple[int, int, int]] = {
    "car": (0, 255, 0),
    "truck": (255, 165, 0),
    "bus": (255, 0, 0),
    "motorcycle": (0, 255, 255),
    "bicycle": (255, 0, 255),
}

_STATE_COLORS: dict[int, tuple[int, int, int]] = {
    0: (0, 200, 0),     # Normal  → green
    1: (0, 200, 255),   # Reduced → yellow/orange
    2: (0, 0, 255),     # Congested → red
    3: (0, 0, 180),     # Accident  → dark red
}


def _create_output_path(video_path: str) -> str:
    """Build output path like legacy: {basename}_VAAET_processed.mp4.

    On Colab: saves to /content/ so it appears in the Files panel.
    On local: saves next to the input video.
    """
    basename = os.path.splitext(os.path.basename(video_path))[0]
    filename = f"{basename}_VAAET_processed.mp4"
    if IN_COLAB:
        return os.path.join("/content", filename)
    else:
        parent = os.path.dirname(os.path.abspath(video_path))
        return os.path.join(parent, filename)


def _draw_annotations(
    frame: np.ndarray,
    active_tracks: list,
    speed_tracker: SmoothedSpeedTracker,
    fps: float,
    frame_h: int,
    global_motion: float,
) -> dict[int, float]:
    """Draw bboxes, type label, and per-vehicle speed on the frame.

    Stationary vehicles are identified FIRST and labelled [S] without
    calling ``estimate_speed`` / ``speed_tracker.update``, preventing
    stale low-speed values from lingering in the smoothing window.

    Returns dict of {track_id: smoothed_speed} for the HUD panel.
    """
    individual_speeds: dict[int, float] = {}
    for track in active_tracks:
        color = _COLORS.get(track.vehicle_type, (255, 255, 255))

        # Bounding box (derive from centroid + fixed half-size as tracker
        # does not expose bbox — matches legacy draw_annotations).
        cx, cy = track.centroid
        half_w, half_h = 50, 35
        cv2.rectangle(frame, (cx - half_w, cy - half_h),
                       (cx + half_w, cy + half_h), color, 2)

        # ── Stationary check FIRST ──
        if is_stationary(track.history):
            # Clear any stale speed history for this track
            speed_tracker.remove_track(track.track_id)
            cv2.putText(frame, "[S]",
                        (cx - half_w, cy - half_h - 12),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.65, (128, 128, 255), 2)
        else:
            # ── Speed estimation only for moving vehicles ──
            speed = estimate_speed(
                track.history, fps=fps, frame_height=frame_h,
                global_motion=global_motion, vehicle_type=track.vehicle_type,
            )
            smoothed = speed_tracker.update(track.track_id, speed)
            if smoothed is not None:
                individual_speeds[track.track_id] = smoothed
                cv2.putText(frame, f"{smoothed:.0f}km/h",
                            (cx - half_w, cy - half_h - 12),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)

        # Type label below box
        cv2.putText(frame, track.vehicle_type.upper(),
                    (cx - half_w, cy + half_h + 22),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65, color, 2)

    return individual_speeds


def _classify_current(records, minute_counts, minute_speeds, label_mapping):
    """Build a temporary telemetry record and classify it.

    Returns ``(state_code, state_label, confidence)`` or ``(None, None, None)``.
    """
    if model is None or scaler is None or label_mapping is None:
        return None, None, None
    try:
        avg_speed = float(np.mean(minute_speeds)) if minute_speeds else 0.0
        total = sum(minute_counts.values())
        tmp_rec = {
            "record_time": datetime.now(),
            "avg_speed": round(avg_speed, 2),
            "count_car": minute_counts.get("car", 0),
            "count_truck": minute_counts.get("truck", 0),
            "count_bus": minute_counts.get("bus", 0),
            "count_motorcycle": minute_counts.get("motorcycle", 0),
            "count_bicycle": minute_counts.get("bicycle", 0),
            "total_vehicles": total,
        }
        # Append temporarily to get feature engineering context (delta, etc.)
        full = records + [tmp_rec]
        df_tmp = pd.DataFrame(full[-2:]) if len(full) >= 2 else pd.DataFrame(full)
        df_feat = engineer_features(df_tmp)
        X = scaler.transform(df_feat[FEATURE_COLS].values[-1:])
        proba = model.predict(X, verbose=0)
        code = int(proba.argmax(axis=1)[0])
        conf = float(proba.max(axis=1)[0])
        lbl = label_mapping.get(code, "Unknown")
        return code, lbl, conf
    except Exception:
        return None, None, None


def _add_info_overlay(
    frame: np.ndarray,
    video_time_s: float,
    avg_speed: float,
    total_counts: dict[str, int],
    current_counts: dict[str, int],
    individual_speeds: dict[int, float],
    state_code: int | None,
    state_label: str | None,
    confidence: float | None,
    frame_idx: int,
    n_active_tracks: int,
) -> np.ndarray:
    """Render full HUD overlay with enlarged, readable text."""
    h, w = frame.shape[:2]

    # ── Left panel background ──
    overlay = frame.copy()
    cv2.rectangle(overlay, (10, 10), (620, 380), (0, 0, 0), -1)
    frame = cv2.addWeighted(frame, 0.7, overlay, 0.3, 0)

    # Timestamp
    hrs = int(video_time_s // 3600)
    mins = int((video_time_s % 3600) // 60)
    secs = int(video_time_s % 60)
    cv2.putText(frame, f"TIME: {hrs:02d}:{mins:02d}:{secs:02d}",
                (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2)

    # Average speed (colour-coded)
    if avg_speed > 80:
        spd_color = (0, 0, 255)       # red  – high
    elif avg_speed > 60:
        spd_color = (0, 165, 255)     # orange – moderate-high
    elif avg_speed < 30:
        spd_color = (255, 255, 0)     # cyan – low
    else:
        spd_color = (0, 255, 255)     # yellow – normal

    cv2.putText(frame, f"AVG SPEED: {avg_speed:.1f} km/h",
                (20, 75), cv2.FONT_HERSHEY_SIMPLEX, 0.8, spd_color, 2)
    cv2.line(frame, (20, 88), (600, 88), (255, 255, 255), 1)

    # Per-type counters
    cv2.putText(frame, "CUMULATIVE COUNTS:",
                (20, 115), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 2)
    y = 140
    total_vehicles = sum(total_counts.values())
    current_total = sum(current_counts.values())

    for vtype in VEHICLE_TYPES:
        tc = total_counts.get(vtype, 0)
        cc = current_counts.get(vtype, 0)
        color = _COLORS.get(vtype, (255, 255, 255))
        if cc > 0:
            txt = f"{vtype.upper()}: {tc} (+{cc})"
            cv2.putText(frame, txt, (35, y),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
            cv2.circle(frame, (18, y - 6), 4, (0, 255, 0), -1)
        else:
            cv2.putText(frame, f"{vtype.upper()}: {tc}", (35, y),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
        y += 24

    total_color = (0, 255, 0) if current_total > 0 else (255, 255, 0)
    cv2.putText(frame, f"TOTAL DETECTED: {total_vehicles}",
                (35, y), cv2.FONT_HERSHEY_SIMPLEX, 0.65, total_color, 2)
    y += 26
    moving = len(individual_speeds)
    stationary = max(current_total - moving, 0)
    if current_total > 0:
        cv2.putText(frame, f"ACTIVE NOW: {current_total}",
                    (35, y), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    else:
        cv2.putText(frame, "NO CURRENT ACTIVITY",
                    (35, y), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (128, 128, 128), 2)
    y += 26
    if stationary > 0:
        cv2.putText(frame, f"STATIONARY: {stationary}",
                    (35, y), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (128, 128, 255), 2)

    # ── Status indicator dot ──
    if moving > 0:
        st_color, st_text = (0, 255, 0), f"ACTIVE ({moving})"
    elif current_total > 0:
        st_color, st_text = (0, 255, 255), f"STATIONARY ({stationary})"
    else:
        st_color, st_text = (0, 0, 255), "NO DETECTIONS"
    cv2.circle(frame, (590, 30), 10, st_color, -1)
    cv2.putText(frame, st_text, (440, 55),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, st_color, 1)

    # ── Right panel: individual speeds ──
    overlay2 = frame.copy()
    if individual_speeds:
        panel_h = min(280, 70 + len(individual_speeds) * 28)
        cv2.rectangle(overlay2, (630, 10), (w - 10, panel_h), (0, 0, 0), -1)
        frame = cv2.addWeighted(frame, 0.7, overlay2, 0.3, 0)
        cv2.putText(frame, f"INDIVIDUAL SPEEDS ({len(individual_speeds)}):",
                    (640, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        sy = 65
        for idx, (tid, spd) in enumerate(individual_speeds.items()):
            if idx >= 8:
                cv2.putText(frame, f"... and {len(individual_speeds) - 8} more",
                            (640, sy), cv2.FONT_HERSHEY_SIMPLEX, 0.45,
                            (128, 128, 128), 1)
                break
            s_col = (0, 0, 255) if spd > 80 else ((255, 255, 0) if spd < 20 else (200, 200, 200))
            cv2.putText(frame, f"#{tid}: {spd:.0f}km/h",
                        (640, sy), cv2.FONT_HERSHEY_SIMPLEX, 0.55, s_col, 1)
            sy += 28
    else:
        cv2.rectangle(overlay2, (630, 10), (w - 10, 90), (0, 0, 0), -1)
        frame = cv2.addWeighted(frame, 0.7, overlay2, 0.3, 0)
        cv2.putText(frame, "NO MOVING", (640, 45),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (128, 128, 128), 2)
        cv2.putText(frame, "VEHICLES", (640, 75),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (128, 128, 128), 2)

    # ── Traffic-state banner (bottom) ──
    banner_h = 70
    if state_code is not None and state_label is not None:
        banner_color = _STATE_COLORS.get(state_code, (100, 100, 100))
        cv2.rectangle(frame, (0, h - banner_h), (w, h), banner_color, -1)
        cv2.putText(frame, f"TRAFFIC STATE: {state_label.upper()}"
                    + (f"  ({confidence:.0%})" if confidence else ""),
                    (20, h - 22), cv2.FONT_HERSHEY_SIMPLEX, 1.0,
                    (255, 255, 255), 2)
    else:
        # Show a muted "awaiting classification" banner so the area isn't blank
        cv2.rectangle(frame, (0, h - banner_h), (w, h), (60, 60, 60), -1)
        cv2.putText(frame, "TRAFFIC STATE: ANALYZING...",
                    (20, h - 22), cv2.FONT_HERSHEY_SIMPLEX, 1.0,
                    (160, 160, 160), 2)

    # ── Tech info (bottom-left above banner) ──
    tech_y = h - banner_h - 15
    cv2.putText(frame, f"Frame: {frame_idx} | Active tracks: {n_active_tracks}",
                (20, tech_y), cv2.FONT_HERSHEY_SIMPLEX, 0.5,
                (128, 128, 128), 1)

    return frame


def generate_annotated_video(
    video_path: str,
    output_path: str | None = None,
    model_variant: str | None = None,
    max_frames: int | None = None,
) -> tuple[str, pd.DataFrame]:
    """Generate annotated video with full HUD, per-minute classification, and
    traffic-state overlay — matching the legacy notebook output.

    Args:
        video_path: Input .mp4 path.
        output_path: Output path.  If ``None``, auto-generated as
            ``{basename}_VAAET_processed.mp4`` in /content/ (Colab) or
            next to the input (local).
        model_variant: YOLO variant (auto-selected if ``None``).
        max_frames: Process at most this many frames (``None`` = full video).

    Returns:
        ``(output_path, df_classified)`` — annotated video path and classified
        DataFrame (one row per minute with traffic_state + confidence).
    """
    # ── Output path ──
    if output_path is None:
        output_path = _create_output_path(video_path)

    # ── Setup ──
    if validate_filename(video_path):
        duration = extract_duration(video_path)
        print(f"📎 Valid bridge filename — duration: {duration:.0f}s")
    else:
        print("⚠️ Non-standard filename — extracting duration from metadata")
        try:
            duration = extract_duration(video_path)
        except ValueError:
            duration = 300.0
            print(f"⚠️ Could not determine duration, using {duration:.0f}s default")

    if model_variant is None:
        model_variant = select_model_variant(duration)

    detector = YOLODetector(model_variant=model_variant)
    detector.load()
    tracker = SORTTracker()
    flow_estimator = OpticalFlowEstimator()
    speed_tracker = SmoothedSpeedTracker(window_size=10)

    cap = open_video(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    frame_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frames_per_minute = int(fps * 60)

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(output_path, fourcc, fps, (frame_w, frame_h))

    # Accumulators
    frame_idx = 0
    minute_counts: dict[str, int] = {v: 0 for v in VEHICLE_TYPES}
    minute_speeds: list[float] = []
    counted_tracks: set[int] = set()
    cumulative_counts: dict[str, int] = {v: 0 for v in VEHICLE_TYPES}

    # Current minute classification state (displayed on HUD)
    cur_state_code: int | None = None
    cur_state_label: str | None = None
    cur_confidence: float | None = None

    # Early classification trigger: classify after EARLY_CLASSIFY_SECS of
    # video so the banner appears quickly (especially for short clips).
    EARLY_CLASSIFY_SECS = 15
    early_classified = False

    records: list[dict] = []

    print(f"\n{'=' * 60}")
    print(f"🌉 VAAET — TRAFFIC ANALYSIS SYSTEM")
    print(f"Puente General Manuel Belgrano")
    print(f"{'=' * 60}")
    print(f"📁 Input:  {os.path.basename(video_path)}")
    print(f"📁 Output: {output_path}")
    print(f"📊 Video:  {duration:.1f}s @ {fps:.1f}fps ({total_frames:,} frames)")
    print(f"📐 Resolution: {frame_w}x{frame_h}")
    print(f"🧠 Model: {model_variant}")
    if max_frames:
        print(f"⚠️  Capped at {max_frames:,} frames")
    print(f"📈 Progress:")

    wall_start = _time.time()
    last_progress_wall = wall_start

    while True:
        ret, frame = cap.read()
        if not ret or (max_frames and frame_idx >= max_frames):
            break

        # 1. Optical flow
        global_motion = flow_estimator.update(frame)

        # 2. Detect
        detections = detector.detect(frame)

        # 3. Track
        det_tuples = [(d.centroid, d.vehicle_type) for d in detections]
        active_tracks = tracker.update(det_tuples)

        # 4. Per-frame counts (for HUD "active now")
        current_frame_counts: dict[str, int] = {v: 0 for v in VEHICLE_TYPES}
        for track in active_tracks:
            current_frame_counts[track.vehicle_type] = (
                current_frame_counts.get(track.vehicle_type, 0) + 1
            )

        # 5. Draw annotations + collect individual speeds
        #    (speed_tracker.update is called ONCE per track inside here;
        #     stationary vehicles are skipped entirely — see _draw_annotations)
        individual_speeds = _draw_annotations(
            frame, active_tracks, speed_tracker, fps, frame_h, global_motion,
        )

        # 6. Accumulate for telemetry
        #    NOTE: speeds already computed in _draw_annotations — do NOT call
        #    speed_tracker.update again (would corrupt the moving average).
        for spd in individual_speeds.values():
            minute_speeds.append(spd)

        for track in active_tracks:
            if track.track_id not in counted_tracks:
                if track.mark_counted():
                    minute_counts[track.vehicle_type] = (
                        minute_counts.get(track.vehicle_type, 0) + 1
                    )
                    counted_tracks.add(track.track_id)

        # ── Early classification trigger ──
        # Classify after EARLY_CLASSIFY_SECS so the banner appears quickly
        # for short clips instead of waiting for the first full minute.
        if (
            not early_classified
            and cur_state_code is None
            and frame_idx >= int(fps * EARLY_CLASSIFY_SECS)
            and (minute_speeds or any(minute_counts.values()))
        ):
            code, lbl, conf = _classify_current(
                records, minute_counts, minute_speeds, label_mapping,
            )
            if code is not None:
                cur_state_code = code
                cur_state_label = lbl
                cur_confidence = conf
                early_classified = True
                print(f"\n   🏷️  Early classification @ {frame_idx / fps:.0f}s: "
                      f"{lbl} ({conf:.0%})")

        # 7. HUD overlay
        avg_speed = float(np.mean(minute_speeds)) if minute_speeds else 0.0
        frame = _add_info_overlay(
            frame,
            video_time_s=frame_idx / fps,
            avg_speed=avg_speed,
            total_counts={v: cumulative_counts.get(v, 0) + minute_counts.get(v, 0)
                          for v in VEHICLE_TYPES},
            current_counts=current_frame_counts,
            individual_speeds=individual_speeds,
            state_code=cur_state_code,
            state_label=cur_state_label,
            confidence=cur_confidence,
            frame_idx=frame_idx,
            n_active_tracks=len(active_tracks),
        )

        writer.write(frame)
        frame_idx += 1

        # ── Progress bar (every 2 s) ──
        wall_now = _time.time()
        if wall_now - last_progress_wall >= 2.0:
            _show_progress_bar(frame_idx, total_frames, wall_now - wall_start)
            last_progress_wall = wall_now

        # ── Stats every 30 s of video time ──
        if frame_idx % int(fps * 30) == 0:
            cum_tot = sum(cumulative_counts.values()) + sum(minute_counts.values())
            active_n = len(active_tracks)
            print(f"\n📊 Stats @ {frame_idx / fps:.0f}s video time:")
            print(f"   📈 Total vehicles so far: {cum_tot}")
            print(f"   ⚡ Current avg speed: {avg_speed:.1f} km/h")
            print(f"   🎯 Active tracks: {active_n}")
            for vt in VEHICLE_TYPES:
                c = cumulative_counts.get(vt, 0) + minute_counts.get(vt, 0)
                if c > 0:
                    print(f"   • {vt.upper()}: {c}")

        # ── Minute boundary: emit telemetry + classify ──
        if frame_idx % frames_per_minute == 0:
            total = sum(minute_counts.values())
            rec = {
                "record_time": datetime.now(),
                "avg_speed": round(avg_speed, 2),
                "count_car": minute_counts.get("car", 0),
                "count_truck": minute_counts.get("truck", 0),
                "count_bus": minute_counts.get("bus", 0),
                "count_motorcycle": minute_counts.get("motorcycle", 0),
                "count_bicycle": minute_counts.get("bicycle", 0),
                "total_vehicles": total,
            }
            records.append(rec)
            print(f"\n   📊 Minute {len(records)}: {avg_speed:.1f} km/h, {total} vehicles")

            # Classify this minute if model is loaded
            if model is not None and scaler is not None and label_mapping is not None:
                try:
                    df_tmp = pd.DataFrame(records[-1:])
                    df_feat = engineer_features(df_tmp)
                    X = scaler.transform(df_feat[FEATURE_COLS].values)
                    proba = model.predict(X, verbose=0)
                    cur_state_code = int(proba.argmax(axis=1)[0])
                    cur_confidence = float(proba.max(axis=1)[0])
                    cur_state_label = label_mapping.get(cur_state_code, "Unknown")
                    print(f"   🏷️  State: {cur_state_label} ({cur_confidence:.0%})")
                except Exception as exc:
                    print(f"   ⚠️ Classification error: {exc}")

            # Accumulate and reset
            for vt in VEHICLE_TYPES:
                cumulative_counts[vt] = cumulative_counts.get(vt, 0) + minute_counts.get(vt, 0)
            minute_counts = {v: 0 for v in VEHICLE_TYPES}
            minute_speeds.clear()
            counted_tracks.clear()

    # ── Flush partial minute ──
    if frame_idx % frames_per_minute != 0 and (minute_speeds or any(minute_counts.values())):
        avg_speed = float(np.mean(minute_speeds)) if minute_speeds else 0.0
        total = sum(minute_counts.values())
        records.append({
            "record_time": datetime.now(),
            "avg_speed": round(avg_speed, 2),
            "count_car": minute_counts.get("car", 0),
            "count_truck": minute_counts.get("truck", 0),
            "count_bus": minute_counts.get("bus", 0),
            "count_motorcycle": minute_counts.get("motorcycle", 0),
            "count_bicycle": minute_counts.get("bicycle", 0),
            "total_vehicles": total,
        })
        # Classify the partial minute too
        if model is not None and scaler is not None and label_mapping is not None:
            try:
                df_tmp = pd.DataFrame(records[-1:])
                df_feat = engineer_features(df_tmp)
                X = scaler.transform(df_feat[FEATURE_COLS].values)
                proba = model.predict(X, verbose=0)
                cur_state_code = int(proba.argmax(axis=1)[0])
                cur_confidence = float(proba.max(axis=1)[0])
                cur_state_label = label_mapping.get(cur_state_code, "Unknown")
                print(f"\n   🏷️  Final partial minute: {cur_state_label} ({cur_confidence:.0%})")
            except Exception as exc:
                print(f"   ⚠️ Partial classification error: {exc}")

        for vt in VEHICLE_TYPES:
            cumulative_counts[vt] = cumulative_counts.get(vt, 0) + minute_counts.get(vt, 0)

    cap.release()
    writer.release()

    # ── Final summary (legacy style) ──
    _show_progress_bar(total_frames, total_frames, _time.time() - wall_start)
    wall_total = _time.time() - wall_start
    grand_total = sum(cumulative_counts.values())

    print(f"\n\n🎉 PROCESSING COMPLETE!")
    print(f"⏱️  Wall time: {wall_total / 60:.1f} min")
    print(f"📊 Frames processed: {frame_idx:,}")
    print(f"🎯 Processing speed: {frame_idx / wall_total:.1f} fps")
    print(f"📈 Detection summary:")
    for vt in VEHICLE_TYPES:
        c = cumulative_counts.get(vt, 0)
        if c > 0:
            print(f"   • {vt.upper()}: {c} vehicles")
    print(f"🚗 Total unique vehicles: {grand_total}")
    print(f"📁 Output: {output_path}")

    # ── Build classified DataFrame ──
    df_records = pd.DataFrame(records)
    df_classified_out: pd.DataFrame = df_records
    if model is not None and scaler is not None and label_mapping is not None and not df_records.empty:
        try:
            df_feat = engineer_features(df_records)
            X = scaler.transform(df_feat[FEATURE_COLS].values)
            proba = model.predict(X, verbose=0)
            df_feat["traffic_state"] = proba.argmax(axis=1)
            df_feat["state_label"] = [label_mapping.get(c, "Unknown") for c in df_feat["traffic_state"]]
            df_feat["confidence"] = proba.max(axis=1).round(4)
            df_classified_out = df_feat
            print("\n✅ Classification summary:")
            for code in sorted(df_classified_out["traffic_state"].unique()):
                lbl = label_mapping.get(code, "Unknown")
                cnt = (df_classified_out["traffic_state"] == code).sum()
                print(f"   {lbl:>10}: {cnt} records")
        except Exception as exc:
            print(f"⚠️ Final classification failed: {exc}")

    # ── Colab: trigger browser download popup ──
    if IN_COLAB and os.path.isfile(output_path):
        from google.colab import files as _colab_dl  # type: ignore[import-untyped]
        print(f"\n📥 Downloading annotated video...")
        _colab_dl.download(output_path)
        print(f"✅ Download complete: {output_path}")

    return output_path, df_classified_out


# ═══════════════════════════════════════════════════════════
# EXECUTION — runs automatically when VIDEO_PATH is set
# ═══════════════════════════════════════════════════════════
try:
    if VIDEO_PATH and os.path.isfile(VIDEO_PATH):
        _out_path, df_classified = generate_annotated_video(VIDEO_PATH)
        # Also set df_telemetry for Cell 3 compatibility
        df_telemetry = df_classified
        print(f"\n📁 Video saved: {_out_path}")
    else:
        print("⚠️ No video file available. Set VIDEO_PATH in Cell 1b or upload a clip.")
        df_telemetry = None
        df_classified = None
except Exception as e:
    print(f"🔴 Error processing clip: {e}")
    import traceback
    traceback.print_exc()
    df_telemetry = None
    df_classified = None

## Classification Pipeline

Takes the per-minute telemetry produced by the perception step, applies the
same feature engineering used during training (via `src.features`), and
classifies each record using the pre-trained MLP model.

In [ ]:
# Cell 3 — Feature Engineering + Classification
#
# NOTE: If you ran Cell 2b, df_classified is already set (classification
# happens inside generate_annotated_video). This cell only needs to run
# if you used Cell 2 (telemetry-only, no video output).

def classify_telemetry(df_telemetry: pd.DataFrame) -> pd.DataFrame:
    """Apply feature engineering and classify traffic state.

    Args:
        df_telemetry: Raw per-minute telemetry from process_clip().

    Returns:
        DataFrame with 14 features + traffic_state + state_label + confidence.
    """
    # Feature engineering (shared with training — 9 → 14 features)
    df_feat = engineer_features(df_telemetry)

    # Scale features
    X = scaler.transform(df_feat[FEATURE_COLS].values)

    # Predict
    proba = model.predict(X, verbose=0)
    pred_codes = proba.argmax(axis=1)
    confidences = proba.max(axis=1)

    df_feat["traffic_state"] = pred_codes
    df_feat["state_label"] = [label_mapping.get(c, "Unknown") for c in pred_codes]
    df_feat["confidence"] = confidences.round(4)

    print("✅ Classification complete:")
    for code in sorted(df_feat["traffic_state"].unique()):
        count = (df_feat["traffic_state"] == code).sum()
        label = label_mapping.get(code, "Unknown")
        print(f"   {label:>10}: {count} records")

    return df_feat


# Execution — skip if Cell 2b already produced df_classified
try:
    if df_classified is not None and "traffic_state" in df_classified.columns:
        print("✅ df_classified already set by Cell 2b — skipping re-classification")
    elif df_telemetry is not None and not df_telemetry.empty:
        df_classified = classify_telemetry(df_telemetry)
    else:
        print("⚠️ No telemetry data — run Cell 2 or Cell 2b first")
        df_classified = None
except NameError:
    print("⚠️ df_telemetry not defined — run Cell 2 or Cell 2b first")
    df_classified = None
except Exception as e:
    print(f"🔴 Classification error: {e}")
    df_classified = None

## Persistence and Feedback

Results are persisted to two PostgreSQL tables:
- **`telemetry_raw`**: The 14 engineered features with FK to original data
- **`traffic_classifications`**: Predictions + confidence + HITL fields

The HITL (Human-in-the-Loop) fields allow operators to validate/override
classifications, creating a feedback loop for model improvement.

In [ ]:
# Cell 4 — Persist Results to Database (Optional)

from sqlalchemy import text as sa_text

DDL_TELEMETRY_RAW = """
CREATE TABLE IF NOT EXISTS telemetry_raw (
    id SERIAL PRIMARY KEY,
    source_record_id INTEGER REFERENCES traffic_data(id),
    record_time TIMESTAMP NOT NULL,
    avg_speed NUMERIC(5,2),
    total_vehicles INTEGER,
    count_car INTEGER, count_truck INTEGER, count_bus INTEGER,
    count_motorcycle INTEGER, count_bicycle INTEGER,
    heavy_vehicle_ratio NUMERIC(5,4),
    delta_speed NUMERIC(6,2), delta_count INTEGER,
    transition_flag SMALLINT DEFAULT 0,
    speed_variance NUMERIC(6,2),
    hour_of_day SMALLINT, weather_condition SMALLINT DEFAULT 0,
    UNIQUE (source_record_id)
);
"""

DDL_TRAFFIC_CLASSIFICATIONS = """
CREATE TABLE IF NOT EXISTS traffic_classifications (
    id SERIAL PRIMARY KEY,
    telemetry_id INTEGER REFERENCES telemetry_raw(id),
    classified_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    traffic_state SMALLINT NOT NULL,
    state_label TEXT NOT NULL,
    confidence NUMERIC(5,4) NOT NULL,
    model_version TEXT NOT NULL,
    is_human_validated BOOLEAN DEFAULT FALSE,
    human_override_state SMALLINT,
    validated_at TIMESTAMP,
    UNIQUE (telemetry_id, model_version)
);
"""

TELEMETRY_COLS: list[str] = [
    "source_record_id", "record_time", "avg_speed", "total_vehicles",
    "count_car", "count_truck", "count_bus", "count_motorcycle",
    "count_bicycle", "heavy_vehicle_ratio", "delta_speed", "delta_count",
    "transition_flag", "speed_variance", "hour_of_day", "weather_condition",
]

INSERT_TELEMETRY_SQL: str = """
    INSERT INTO telemetry_raw (
        source_record_id, record_time, avg_speed, total_vehicles,
        count_car, count_truck, count_bus, count_motorcycle,
        count_bicycle, heavy_vehicle_ratio, delta_speed, delta_count,
        transition_flag, speed_variance, hour_of_day, weather_condition
    ) VALUES (
        :source_record_id, :record_time, :avg_speed, :total_vehicles,
        :count_car, :count_truck, :count_bus, :count_motorcycle,
        :count_bicycle, :heavy_vehicle_ratio, :delta_speed, :delta_count,
        :transition_flag, :speed_variance, :hour_of_day, :weather_condition
    )
    ON CONFLICT (source_record_id) DO NOTHING
"""

INSERT_CLASSIFICATION_SQL: str = """
    INSERT INTO traffic_classifications (
        telemetry_id, traffic_state, state_label,
        confidence, model_version
    ) VALUES (
        :telemetry_id, :traffic_state, :state_label,
        :confidence, :model_version
    )
    ON CONFLICT (telemetry_id, model_version) DO NOTHING
"""


def persist_classifications(df: pd.DataFrame, config: dict[str, str]) -> None:
    """Persist classified telemetry to PostgreSQL.

    Steps:
      1. Create tables if they don't exist.
      2. Batch INSERT into ``telemetry_raw`` (14 features per record).
      3. Batch INSERT into ``traffic_classifications`` (state + confidence).

    Args:
        df: DataFrame with features + ``traffic_state`` + ``confidence``
            columns.  Must also contain an ``id`` column mapping back to
            ``traffic_data.id`` (used as ``source_record_id``).
        config: Database credentials dict.
    """
    engine = get_engine(config)

    with engine.begin() as conn:
        conn.execute(sa_text(DDL_TELEMETRY_RAW))
        conn.execute(sa_text(DDL_TRAFFIC_CLASSIFICATIONS))
        print("✅ Tables created (or already exist)")

        # telemetry_raw batch INSERT
        df_telemetry = df.rename(columns={"id": "source_record_id"})[
            [c for c in TELEMETRY_COLS if c in df.columns or c == "source_record_id"]
        ].copy()

        # Ensure correct types for PostgreSQL
        for int_col in ("delta_count", "transition_flag", "hour_of_day", "weather_condition"):
            if int_col in df_telemetry.columns:
                df_telemetry[int_col] = df_telemetry[int_col].astype(int)

        telemetry_records = df_telemetry.to_dict(orient="records")
        conn.execute(sa_text(INSERT_TELEMETRY_SQL), telemetry_records)
        print(f"📊 telemetry_raw: {len(telemetry_records)} records sent")

        # Map source_record_id → telemetry_raw.id
        telemetry_ids = conn.execute(
            sa_text("SELECT id, source_record_id FROM telemetry_raw ORDER BY id")
        ).fetchall()
        source_to_telemetry = {row[1]: row[0] for row in telemetry_ids}

        # traffic_classifications batch INSERT
        classification_records: list[dict] = []
        for _, row in df.iterrows():
            source_id = row.get("id")
            telemetry_id = source_to_telemetry.get(source_id)
            if telemetry_id is None:
                continue
            classification_records.append({
                "telemetry_id": int(telemetry_id),
                "traffic_state": int(row["traffic_state"]),
                "state_label": STATE_LABELS[int(row["traffic_state"])],
                "confidence": float(round(row.get("confidence", 0.0), 4)),
                "model_version": MODEL_VERSION,
            })

        if classification_records:
            conn.execute(sa_text(INSERT_CLASSIFICATION_SQL), classification_records)

        print(f"📊 traffic_classifications: {len(classification_records)} records sent")

    engine.dispose()

    # Summary
    print(f"\n✅ Persistence completed (model_version={MODEL_VERSION})")
    dist = df["traffic_state"].value_counts().sort_index()
    for code, count in dist.items():
        label = STATE_LABELS.get(code, f"State {code}")
        print(f"   {label:>10}: {count} classifications")


# ── Execution (DB is optional — mirrors M1 pattern) ──
try:
    db_config = get_db_config()
except Exception:
    db_config = None

if db_config is not None:
    try:
        if df_classified is not None and not df_classified.empty:
            persist_classifications(df_classified, db_config)
        else:
            print("⚠️ Run Cells 2-3 first, then re-run this cell")
    except NameError:
        print("⚠️ Run Cells 2-3 first, then re-run this cell")
    except Exception as e:
        print(f"🔴 Persistence error: {e}")
        print("   Classified data is available in-memory (df_classified)")
else:
    print("⚠️ No DB configured — skipping persistence (data stays in-memory as df_classified)")
    print("   Set DB_HOST / DB_NAME / DB_USER / DB_PASSWORD env vars to enable.")

## Feedback Loop — Re-training

This cell implements the self-improvement cycle:
1. Load human-validated records from `traffic_classifications` (where `is_human_validated = TRUE`)
2. Merge with original training data
3. Re-train the MLP with the expanded dataset
4. Export updated `.keras` artifact

This closes the feedback loop: **production → HITL validation → re-training → better production**.

In [ ]:
# Cell 5 — Feedback Loop: Re-train with HITL Data (Optional)

from sklearn.preprocessing import StandardScaler as _StandardScaler
from sklearn.model_selection import train_test_split as _train_test_split
from sklearn.metrics import f1_score as _f1_score
from imblearn.over_sampling import SMOTE as _SMOTE


def retrain_with_feedback(config: dict[str, str]) -> None:
    """Re-train the classifier using human-validated data.

    Loads validated classifications from the database, merges them with
    the original training data, and re-trains the MLP model. The updated
    model is exported only if F1-macro improves over the current one.

    Args:
        config: Database credentials.
    """
    engine = get_engine(config)

    # Load human-validated records
    query = """
        SELECT tr.*, tc.traffic_state AS validated_state
        FROM telemetry_raw tr
        JOIN traffic_classifications tc ON tc.telemetry_id = tr.id
        WHERE tc.is_human_validated = TRUE
        ORDER BY tr.record_time
    """
    df_validated = pd.read_sql(sa_text(query), engine)
    engine.dispose()

    if df_validated.empty:
        print("⚠️ No human-validated records found. Skipping re-training.")
        return

    print(f"📊 Loaded {len(df_validated)} validated records")

    # Merge with original training data 
    _root = os.path.join("..", "..")
    csv_path = os.path.join(_root, "data", "processed", "traffic_telemetry.csv")
    if not os.path.exists(csv_path):
        print("🔴 Original training CSV not found. Run Module 1 first.")
        return

    df_original = pd.read_csv(csv_path)
    if "traffic_state" not in df_original.columns:
        df_original["traffic_state"] = assign_traffic_state(df_original)

    # Use validated_state as ground truth for HITL records
    df_validated_features = df_validated[FEATURE_COLS].copy()
    df_validated_features["traffic_state"] = df_validated["validated_state"].astype(int)

    df_combined = pd.concat(
        [df_original[FEATURE_COLS + ["traffic_state"]], df_validated_features],
        ignore_index=True,
    )
    print(f"📊 Combined dataset: {len(df_combined)} records "
          f"({len(df_original)} original + {len(df_validated)} validated)")

    # Split (raw, unscaled) 
    X_raw = df_combined[FEATURE_COLS].values
    y = df_combined["traffic_state"].values

    X_train_raw, X_test_raw, y_train, y_test = _train_test_split(
        X_raw, y, test_size=0.2, stratify=y, random_state=RANDOM_SEED,
    )

    # Evaluate old model on test set (using its own scaler) 
    X_test_old = scaler.transform(X_test_raw)
    y_pred_old = model.predict(X_test_old, verbose=0).argmax(axis=1)
    f1_old = _f1_score(y_test, y_pred_old, average="macro", zero_division=0)

    # Fit new scaler + SMOTE on training data
    new_scaler = _StandardScaler()
    X_train_new = new_scaler.fit_transform(X_train_raw)
    X_test_new = new_scaler.transform(X_test_raw)

    train_counts = np.bincount(y_train)
    min_class = train_counts[train_counts > 0].min()
    k_neighbors = min(5, min_class - 1) if min_class > 1 else 1

    if min_class >= 2:
        sm = _SMOTE(random_state=RANDOM_SEED, k_neighbors=k_neighbors)
        X_train_res, y_train_res = sm.fit_resample(X_train_new, y_train)
        print(f"✅ SMOTE applied (k_neighbors={k_neighbors})")
    else:
        X_train_res, y_train_res = X_train_new, y_train
        print("⚠️ SMOTE skipped — class with <2 samples")

    # Re-train MLP
    from tensorflow.keras.models import Sequential as _Sequential
    from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input
    from tensorflow.keras.callbacks import EarlyStopping

    n_classes = len(np.unique(y))
    new_model = _Sequential([
        Input(shape=(X_train_res.shape[1],)),
        Dense(64, activation="relu"),
        BatchNormalization(),
        Dropout(0.3),
        Dense(32, activation="relu"),
        BatchNormalization(),
        Dropout(0.2),
        Dense(n_classes, activation="softmax"),
    ], name="traffic_state_classifier_retrained")

    new_model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    new_model.fit(
        X_train_res, y_train_res,
        epochs=200,
        batch_size=32,
        validation_split=0.2,
        callbacks=[EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True)],
        verbose=1,
    )

    # Evaluate new model on test set
    y_pred_new = new_model.predict(X_test_new, verbose=0).argmax(axis=1)
    f1_new = _f1_score(y_test, y_pred_new, average="macro", zero_division=0)

    print(f"\n📊 F1-macro comparison:")
    print(f"   Current model: {f1_old:.4f}")
    print(f"   Retrained:     {f1_new:.4f}")

    # Export if improved
    if f1_new > f1_old:
        model_path = os.path.join(_root, "models", "intelligence", "traffic_classifier.keras")
        scaler_path = os.path.join(_root, "models", "intelligence", "feature_scaler.joblib")
        new_model.save(model_path)
        joblib.dump(new_scaler, scaler_path)
        print(f"✅ Improved model exported → {model_path}")
        print(f"   New scaler exported → {scaler_path}")
    else:
        print("⚠️ Retrained model did not improve. Keeping current model.")


# Execution
# Uncomment when HITL data is available in traffic_classifications:
# retrain_config = get_db_config()
# retrain_with_feedback(retrain_config)
print("⚠️ Re-training requires human-validated data in traffic_classifications")

## Visualization

Summary dashboard showing traffic state distribution, speed timeline, and
classification confidence. Only runs after Cells 2-3 have been executed.

In [ ]:
# Cell 6 — Visualization Dashboard

import matplotlib.pyplot as plt

def show_dashboard(df: pd.DataFrame) -> None:
    """Display a 5-panel summary dashboard for the classified telemetry.

    Panels:
      1. Traffic state distribution (bar)
      2. Average speed over time (line)
      3. Classification confidence histogram
      4. Vehicle counts by type (stacked area)
      5. Speed vs. total vehicles scatter

    Args:
        df: Classified DataFrame with traffic_state, avg_speed, confidence,
            and per-type count columns.
    """
    fig = plt.figure(figsize=(20, 10))
    colors = ["#2ecc71", "#f39c12", "#e74c3c", "#8e44ad"]
    type_colors = {
        "car": "#3498db", "truck": "#e67e22", "bus": "#e74c3c",
        "motorcycle": "#2ecc71", "bicycle": "#9b59b6",
    }

    # Panel 1: State distribution
    ax1 = fig.add_subplot(2, 3, 1)
    dist = df["traffic_state"].value_counts().sort_index()
    state_names = [label_mapping.get(c, f"State {c}") for c in sorted(dist.index)]
    state_colors = [colors[c] for c in sorted(dist.index)]
    ax1.bar(state_names, dist.values, color=state_colors)
    ax1.set_title("Traffic State Distribution")
    ax1.set_ylabel("Records")

    # Panel 2: Speed timeline
    ax2 = fig.add_subplot(2, 3, 2)
    x_axis = range(len(df))
    ax2.plot(x_axis, df["avg_speed"], color="#3498db", linewidth=1.5, label="Avg Speed")
    if "speed_variance" in df.columns:
        ax2.fill_between(
            x_axis,
            df["avg_speed"] - df["speed_variance"].clip(lower=0).pow(0.5),
            df["avg_speed"] + df["speed_variance"].clip(lower=0).pow(0.5),
            alpha=0.2, color="#3498db", label="±1 σ",
        )
    ax2.set_title("Average Speed Over Time")
    ax2.set_xlabel("Minute")
    ax2.set_ylabel("Speed (km/h)")
    ax2.legend(fontsize=8)
    ax2.grid(True, alpha=0.3)

    # Panel 3: Confidence distribution
    ax3 = fig.add_subplot(2, 3, 3)
    ax3.hist(df["confidence"], bins=20, color="#9b59b6", edgecolor="white")
    ax3.axvline(0.8, color="#e74c3c", linestyle="--", linewidth=1, label="Threshold 0.8")
    ax3.set_title("Classification Confidence")
    ax3.set_xlabel("Confidence")
    ax3.set_ylabel("Frequency")
    ax3.legend(fontsize=8)

    # Panel 4: Vehicle counts by type (stacked area)
    ax4 = fig.add_subplot(2, 3, 4)
    count_cols = [c for c in ["count_car", "count_truck", "count_bus",
                               "count_motorcycle", "count_bicycle"]
                  if c in df.columns]
    if count_cols:
        df_counts = df[count_cols].fillna(0)
        ax4.stackplot(
            x_axis, *[df_counts[c] for c in count_cols],
            labels=[c.replace("count_", "") for c in count_cols],
            colors=[type_colors.get(c.replace("count_", ""), "#999") for c in count_cols],
            alpha=0.8,
        )
        ax4.set_title("Vehicle Counts by Type")
        ax4.set_xlabel("Minute")
        ax4.set_ylabel("Count")
        ax4.legend(loc="upper left", fontsize=7)
    else:
        ax4.text(0.5, 0.5, "No count data", ha="center", va="center")
        ax4.set_title("Vehicle Counts by Type")

    # Panel 5: Speed vs. total vehicles (scatter)
    ax5 = fig.add_subplot(2, 3, 5)
    if "total_vehicles" in df.columns:
        scatter_colors = [colors[c] if c < len(colors) else "#999"
                          for c in df["traffic_state"]]
        ax5.scatter(df["total_vehicles"], df["avg_speed"], c=scatter_colors,
                    alpha=0.7, edgecolors="white", linewidth=0.5)
        ax5.set_xlabel("Total Vehicles")
        ax5.set_ylabel("Avg Speed (km/h)")
        ax5.set_title("Speed vs. Volume")
        ax5.grid(True, alpha=0.3)
    else:
        ax5.text(0.5, 0.5, "No volume data", ha="center", va="center")
        ax5.set_title("Speed vs. Volume")

    # Panel 6: Summary text panel
    ax6 = fig.add_subplot(2, 3, 6)
    ax6.axis("off")
    summary_lines = [
        f"📊 Total records: {len(df)}",
        f"⏱️  Avg speed: {df['avg_speed'].mean():.1f} km/h",
        f"📈 Max speed: {df['avg_speed'].max():.1f} km/h",
        f"📉 Min speed: {df['avg_speed'].min():.1f} km/h",
    ]
    if "total_vehicles" in df.columns:
        summary_lines.append(f"🚗 Total vehicles: {df['total_vehicles'].sum():.0f}")
    if "confidence" in df.columns:
        summary_lines.append(f"🎯 Avg confidence: {df['confidence'].mean():.3f}")
        low_conf = (df["confidence"] < 0.8).sum()
        summary_lines.append(f"⚠️  Low confidence (<0.8): {low_conf}")
    summary_text = "\n".join(summary_lines)
    ax6.text(0.1, 0.5, summary_text, fontsize=11, verticalalignment="center",
             fontfamily="monospace", transform=ax6.transAxes)
    ax6.set_title("Summary")

    plt.tight_layout()
    plt.show()


# Execution — requires df_classified from Cell 3
try:
    if df_classified is not None and not df_classified.empty:
        show_dashboard(df_classified)
    else:
        print("⚠️ No classified data — run Cells 2-3 first")
except NameError:
    print("⚠️ df_classified not defined — run Cells 2-3 first")
except Exception as e:
    print(f"🔴 Dashboard error: {e}")